In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from tensorflow.keras.models import load_model
import numpy as np
import cv2
from tensorflow.keras.applications.vgg16 import preprocess_input
import random

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report


In [ ]:

model_type = load_model("/content/drive/MyDrive/ProjectData/BrainTumor/experiments/task1_customCNN_aug_best.keras",custom_objects={'preprocess_input': preprocess_input})

In [ ]:


model_grade = load_model("/content/drive/MyDrive/ProjectData/combined_splits/vgg16_baseline_grade_aug_best.keras",
                         custom_objects={'preprocess_input': preprocess_input})


In [ ]:
IMG_SIZE = (224, 224)

def preprocess_image(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, IMG_SIZE)
    img = img.astype("float32") / 255.0
    return np.expand_dims(img, axis=0)


In [ ]:
class TumorPipeline:
    def __init__(self, type_model, grade_model):
        self.type_model = type_model
        self.grade_model = grade_model
        self.type_classes = ["glioma", "meningioma", "pituitary"]

    def predict(self, img_path):
        x = preprocess_image(img_path)

        # Stage 1: tumor type prediction
        type_pred = self.type_model.predict(x)
        type_label = self.decode_type(type_pred)

        # Stage 2: only if glioma → do grade classification
        if type_label == "glioma":
            grade_pred = self.grade_model.predict(x)
            grade_label = self.decode_grade(grade_pred)

            return {
                "Tumor Type": type_label,
                "Grade": grade_label,
                "Raw_Type_Output": type_pred,
                "Raw_Grade_Output": grade_pred
            }
        else:
            return {
                "Tumor Type": type_label,
                "Grade": None,
                "Raw_Type_Output": type_pred
            }

    def decode_type(self, pred):
        return self.type_classes[np.argmax(pred)]

    def decode_grade(self, pred):
        return "High Grade" if pred[0][0] > 0.5 else "Low Grade"


In [ ]:
test_folder = "/content/drive/MyDrive/ProjectData/BrainTumor/Testing"


In [ ]:
import os
import random

test_folder = "/content/drive/MyDrive/ProjectData/BrainTumor/Testing"

# Get all subfolders (glioma, meningioma, notumor, pituitary)
class_folders = [os.path.join(test_folder, d) for d in os.listdir(test_folder)
                 if os.path.isdir(os.path.join(test_folder, d))]

# Randomly select one class
chosen_class_folder = random.choice(class_folders)

# List all image files inside that class
image_files = [f for f in os.listdir(chosen_class_folder)
               if f.lower().endswith((".jpg", ".jpeg", ".png"))]

if len(image_files) == 0:
    raise Exception(f"No images found in folder: {chosen_class_folder}")

# Pick random image
random_image = random.choice(image_files)
random_path = os.path.join(chosen_class_folder, random_image)

print("Randomly selected image:", random_path)


Randomly selected image: /content/drive/MyDrive/ProjectData/BrainTumor/Testing/meningioma/Te-me_0093.jpg


In [ ]:
pipeline = TumorPipeline(model_type, model_grade)


In [ ]:
result = pipeline.predict(random_path)

print("\nPrediction Results")
print("Tumor Type:", result["Tumor Type"])

if result["Grade"] is not None:
    print("Tumor Grade:", result["Grade"])
else:
    print("Grade not required for this tumor type.")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step

Prediction Results
Tumor Type: meningioma
Grade not required for this tumor type.
